In [ ]:
!apt-get update
!apt-get install -y fonts-dejavu
!apt-get install -y fontconfig
!fc-cache -fv
!pip install tqdm

import os
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import zipfile
from io import BytesIO
import re
from tqdm import tqdm
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
def fix_tsv_file(input_tsv_path, output_tsv_path):
    """
    исправляет криво отформатированный tsv файл со статистикой символов
    """

    with open(input_tsv_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    fixed_lines = ["unit\tcount\tpercent\n"]
    problem_lines = []

    for i, line in enumerate(lines[1:], 2):
        line = line.strip()
        if not line:
            continue

        parts = line.split()

        if len(parts) >= 3:
            # последние две части должны быть числами (count и percent)
            try:
                count = parts[-2]
                percent = parts[-1]
                float(count)
                float(percent)

                unit = ' '.join(parts[:-2])

                fixed_lines.append(f"{unit}\t{count}\t{percent}\n")
            except ValueError:
                problem_lines.append((i, line))
        else:
            problem_lines.append((i, line))

    with open(output_tsv_path, 'w', encoding='utf-8') as f:
        f.writelines(fixed_lines)

    return fixed_lines


def verify_stats_file(file_path):
    """
    проверяет корректность tsv файла
    """

    chars = []
    counts = []
    percents = []
    total_percent = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        header = f.readline().strip()

        for i, line in enumerate(f, 2):
            line = line.strip()
            if not line:
                continue

            parts = line.split('\t')
            if len(parts) == 3:
                unit, count, percent = parts
                try:
                    count_val = int(count)
                    percent_val = float(percent)

                    chars.append(unit)
                    counts.append(count_val)
                    percents.append(percent_val)
                    total_percent = total_percent + percent_val
                except ValueError as e:
                    print(f"ошибка в строке {i}: {line} - {e}")
            else:
                print(f"неверный формат в строке {i}: {line}")

    return chars, counts, percents


input_tsv = "/content/drive/MyDrive/4_Adyghe_NoOCR_balanced_char_stats.tsv"
output_tsv = "/content/drive/MyDrive/Adyghe_NoOCR_balanced_char_stats_fixed.tsv"

fixed_lines = fix_tsv_file(input_tsv, output_tsv)

chars, counts, percents = verify_stats_file(output_tsv)

In [ ]:
def load_char_stats_simple(tsv_path):
    """загружает статистику символов из исправленного tsv файла"""
    char_percent = {}

    with open(tsv_path, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) == 3:
                unit, count, percent = parts
                try:
                    char_percent[unit] = float(percent)
                except ValueError:
                    continue

    # нормализуем проценты
    total = sum(char_percent.values())
    if abs(total - 100) > 0.1:
        for char in char_percent:
            char_percent[char] = (char_percent[char] / total) * 100

    return char_percent


def calculate_char_distribution(texts):
    """рассчитывает распределение символов в списке текстов"""
    total_chars = 0
    char_counts = Counter()

    for text in texts:
        total_chars = total_chars + len(text)
        char_counts.update(text)

    distribution = {}
    for char, count in char_counts.items():
        distribution[char] = (count / total_chars) * 100 if total_chars > 0 else 0

    return distribution, char_counts, total_chars


def random_sentence_selection(input_text_path, target_sentences=100000,
                              min_len=80, max_len=100):
    with open(input_text_path, 'r', encoding='utf-8') as f:
        full_text = f.read()

    print(f"исходный текст: {len(full_text):,} символов")

    sentences = re.split(r'(?<=[.!?])\s+', full_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    print(f"всего предложений: {len(sentences):,}")

    # сначала фильтруем по длине
    valid_sentences = [s for s in sentences if min_len <= len(s) <= max_len]

    # дополнительно фильтруем предложения с тремя и более дефисами подряд
    valid_sentences = [s for s in valid_sentences if not re.search(r'-{3,}', s)]

    print(f"предложений подходящей длины: {len(valid_sentences):,}")

    # проверяем, достаточно ли предложений
    if len(valid_sentences) < target_sentences:
        target_sentences = len(valid_sentences)

    random.seed(42)
    selected_sentences = random.sample(valid_sentences, target_sentences)

    random.shuffle(selected_sentences)

    # проверяем распределение по длине
    lengths = [len(s) for s in selected_sentences]
    print(f"минимум: {min(lengths)} символов")
    print(f"максимум: {max(lengths)} символов")
    print(f"среднее: {sum(lengths)/len(lengths):.1f} символов")

    return selected_sentences

TARGET_SENTENCES = 100000
MIN_LEN = 80
MAX_LEN = 100
RANDOM_SEED = 42

target_stats = load_char_stats_simple('/content/drive/MyDrive/Adyghe_NoOCR_balanced_char_stats_fixed.tsv')

selected_sentences = random_sentence_selection(
    input_text_path='/content/drive/MyDrive/4_Adyghe_NoOCR_balanced.txt',
    target_sentences=TARGET_SENTENCES,
    min_len=MIN_LEN,
    max_len=MAX_LEN
)

output_text_path = '/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled.txt'
with open(output_text_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(selected_sentences))

final_dist, final_counts, final_total = calculate_char_distribution(selected_sentences)

stats_output = '/content/drive/MyDrive/Adyghe_NoOCR_balanced_char_stats_fixed.tsv'
with open(stats_output, 'w', encoding='utf-8') as f:
    f.write("unit\tcount\tpercent\n")

    sorted_items = sorted(final_counts.items(), key=lambda x: x[1], reverse=True)
    for char, count in sorted_items:
        percent = (count / final_total) * 100
        if char in ['\t', '\n', ' ']:
            char_repr = f"'{char}'"
        else:
            char_repr = char
        f.write(f"{char_repr}\t{count}\t{percent:.6f}\n")

исходный текст: 95,131,923 символов
всего предложений: 1,002,516
предложений подходящей длины: 131,451
минимум: 80 символов
максимум: 100 символов
среднее: 89.6 символов


In [ ]:
def create_chunks_from_sampled_text(input_file_path, output_zip_path, max_chunks=100000):
    """
    создает чанки из уже отобранных предложений
    """
    try:
        with open(input_file_path, 'r', encoding='utf-8') as f:
            full_text = f.read()
        print(f"исходный файл прочитан: {len(full_text):,} символов")
    except Exception as e:
        print(e)
        return

    # разбиваем на предложения
    chunks = full_text.split('\n')
    chunks = [c for c in chunks if c.strip()]
    print(f"всего предложений в файле: {len(chunks):,}")

    if len(chunks) > max_chunks:
        random.seed(42)
        chunks = random.sample(chunks, max_chunks)

    try:
        with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for i, chunk in enumerate(chunks, 1):
                filename = f"{i:06d}.txt"
                zipf.writestr(filename, chunk)

                if i % 25000 == 0:
                    print(f"обработано {i:,} файлов...")

    except Exception as e:
        print(e)

    return chunks


INPUT_FILE = "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled.txt"
OUTPUT_ZIP = "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled.zip"
MAX_CHUNKS = 100000

chunks = create_chunks_from_sampled_text(
    input_file_path=INPUT_FILE,
    output_zip_path=OUTPUT_ZIP,
    max_chunks=MAX_CHUNKS
)

print(f"итоговое количество чанков: {len(chunks) if chunks else 0:,}")

исходный файл прочитан: 9,064,823 символов
всего предложений в файле: 100,000
обработано 25,000 файлов...
обработано 50,000 файлов...
обработано 75,000 файлов...
обработано 100,000 файлов...
итоговое количество чанков: 100,000


In [ ]:
LANG_PREFIX = "Adyghe_NoOCR_balanced_sampled"

FONT_SIZE = 20
DPI = 80
LINE_SPACING = 1.0
MARGIN_PX = 7

HAS_SKEW = [0, 1]        # 0 - нет, 1 - есть
HAS_THICKNESS = [0, 1]   # 0 - нет, 1 - есть

TMP_DIR = "/content/tmp_txt"


def get_cyrillic_fonts():
    return {
        "Sans": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "Serif": "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
        "Mono": "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
    }


def extract_part_counter(filename):
    nums = re.findall(r"\d+", filename)
    if not nums:
        raise ValueError(f"нет числа в имени файла: {filename}")
    return int(nums[0])


def split_text_into_lines_by_width(text, font, max_width):
    words = text.split()
    lines = []
    current = []

    for word in words:
        current.append(word)
        bbox = font.getbbox(" ".join(current))
        if bbox[2] > max_width:
            current.pop()
            lines.append(" ".join(current))
            current = [word]

    if current:
        lines.append(" ".join(current))

    return lines


def create_background(width, height):
    base = Image.new("RGB", (width, height), (245, 245, 245))
    noise = np.random.normal(0, 1.0, (height, width, 3))
    arr = np.clip(np.array(base) + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def add_scan_distortions(image, intensity=0.25):
    arr = np.array(image).astype(np.float32)

    # лёгкий шум
    if random.random() < intensity:
        arr = arr + np.random.normal(0, random.uniform(0.5, 1.2), arr.shape)

    # слабая яркость
    if random.random() < 0.3 * intensity:
        arr = arr * random.uniform(0.97, 1.03)

    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def generate_from_txt_zip(
    txt_zip_path,
    images_zip_path,
    gt_zip_path,
    file_slice=(0, None),
    start_index=1  # добавляем параметр начального индекса
):
    fonts = get_cyrillic_fonts()
    os.makedirs(TMP_DIR, exist_ok=True)

    with zipfile.ZipFile(txt_zip_path, "r") as zin:
        txt_files = sorted([f for f in zin.namelist() if f.endswith(".txt")])
        txt_files = txt_files[file_slice[0]:file_slice[1]]
        zin.extractall(TMP_DIR)

    # сохраняем во временные папки
    temp_images_dir = "/content/temp_images"
    temp_gt_dir = "/content/temp_gt"
    os.makedirs(temp_images_dir, exist_ok=True)
    os.makedirs(temp_gt_dir, exist_ok=True)

    processed = 0
    total = len(txt_files)

    for txt_idx, txt_name in enumerate(tqdm(txt_files, desc="обработка файлов")):
        part_counter = extract_part_counter(txt_name)

        with open(os.path.join(TMP_DIR, txt_name), encoding="utf-8") as f:
            text = re.sub(r"\s+", " ", f.read()).strip()

        for font_label, font_path in fonts.items():
            font = ImageFont.truetype(font_path, FONT_SIZE)

            max_width = 2000
            lines = split_text_into_lines_by_width(text, font, max_width)

            line_height = int(FONT_SIZE * LINE_SPACING)
            text_height = line_height * len(lines)
            text_width = max(font.getbbox(line)[2] for line in lines)

            img_w = text_width + 2 * MARGIN_PX
            img_h = text_height + 2 * MARGIN_PX

            for skew in HAS_SKEW:
                for thick in HAS_THICKNESS:

                    img = create_background(img_w, img_h)
                    draw = ImageDraw.Draw(img)

                    y = MARGIN_PX
                    for ln in lines:
                        if thick:
                            draw.text(
                                (MARGIN_PX + 1, y + 1),
                                ln,
                                font=font,
                                fill=(90, 90, 90)
                            )
                        draw.text(
                            (MARGIN_PX, y),
                            ln,
                            font=font,
                            fill=(0, 0, 0)
                        )
                        y = y + line_height

                    if skew:
                        img = img.rotate(
                            random.uniform(-1.2, 1.2),
                            expand=True,
                            fillcolor=(245, 245, 245)
                        )

                    img = add_scan_distortions(img, intensity=0.25)

                    # используем start_index + txt_idx для непрерывной нумерации
                    file_number = start_index + txt_idx

                    base_name = (
                        f"{LANG_PREFIX}_"
                        f"{file_number:06d}_"
                        f"{font_label}_"
                        f"{skew}_"
                        f"{thick}"
                    )

                    img_path = os.path.join(temp_images_dir, f"{base_name}.jpg")
                    img.save(img_path, format="JPEG", quality=80, dpi=(DPI, DPI), optimize=True)

                    gt_path = os.path.join(temp_gt_dir, f"{base_name}.gt.txt")
                    with open(gt_path, 'w', encoding='utf-8') as gt_f:
                        gt_f.write(text)

        processed = processed + 1

        if processed % 1000 == 0:
            print(f"обработано {processed}/{total} текстов")

    with zipfile.ZipFile(images_zip_path, "w", zipfile.ZIP_DEFLATED) as images_zip:
        for file in tqdm(sorted(os.listdir(temp_images_dir)), desc="добавление изображений"):
            file_path = os.path.join(temp_images_dir, file)
            images_zip.write(file_path, file)

    with zipfile.ZipFile(gt_zip_path, "w", zipfile.ZIP_DEFLATED) as gt_zip:
        for file in tqdm(sorted(os.listdir(temp_gt_dir)), desc="добавление GT"):
            file_path = os.path.join(temp_gt_dir, file)
            gt_zip.write(file_path, file)

    shutil.rmtree(temp_images_dir)
    shutil.rmtree(temp_gt_dir)

    img_size = os.path.getsize(images_zip_path) / (1024*1024)
    gt_size = os.path.getsize(gt_zip_path) / (1024*1024)
    print(f"{os.path.basename(images_zip_path)}: {img_size:.2f} MB")
    print(f"{os.path.basename(gt_zip_path)}: {gt_size:.2f} MB")

In [ ]:
generate_from_txt_zip(
    txt_zip_path="/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled.zip",
    images_zip_path="/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART10.zip",
    gt_zip_path="/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART10.zip",
    file_slice=(90000, 100000),
    start_index=90001
)

# file_slice=(0, 10000) part 1 - ок
# file_slice=(10000, 20000) part2 - ок
# file_slice=(20000, 30000) part3 - ок
# file_slice=(30000, 40000) part4 - ок
# file_slice=(40000, 50000) part5 - ок
# file_slice=(50000, 60000) part6 - ок
# file_slice=(60000, 70000), part7 - ок
# file_slice=(70000, 80000), part8 - ок
# file_slice=(80000, 90000), part9 - ок
# file_slice=(90000, 100000), part10 - ок

In [ ]:
def merge_and_clean_images(part_zips, output_zip_path, temp_dir="/content/temp_merge"):
    """
    объединяет zip части и удаляет их после обработки
    """
    # создаем временную папку
    os.makedirs(temp_dir, exist_ok=True)

    total_files = 0
    processed_parts = []

    try:
        with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as out_zip:
            for i, part_path in enumerate(part_zips, 1):
                print(f"\nчасть {i}/{len(part_zips)}: {os.path.basename(part_path)}")

                if not os.path.exists(part_path):
                    continue

                part_size = os.path.getsize(part_path) / (1024*1024)
                print(f"размер части: {part_size:.2f} MB")

                part_temp = os.path.join(temp_dir, f"part_{i}")
                os.makedirs(part_temp, exist_ok=True)

                with zipfile.ZipFile(part_path, 'r') as part_zip:
                    files = part_zip.namelist()
                    print(f"файлов в части: {len(files)}")

                    for filename in tqdm(files, desc="распаковка", leave=False):
                        part_zip.extract(filename, part_temp)

                extracted_files = os.listdir(part_temp)
                for filename in tqdm(extracted_files, desc="добавление", leave=False):
                    file_path = os.path.join(part_temp, filename)
                    out_zip.write(file_path, filename)
                    total_files = total_files + 1

                # очистка временных файлов
                shutil.rmtree(part_temp)

                os.remove(part_path)
                processed_parts.append(part_path)

                current_size = os.path.getsize(output_zip_path) / (1024*1024)
                print(f"текущий размер общего архива: {current_size:.2f} MB")
                print(f"часть {i} обработана и удалена")

        final_size = os.path.getsize(output_zip_path) / (1024*1024)
        print(f"итоговый zip: {output_zip_path}")
        print(f"размер: {final_size:.2f} MB")
        print(f"всего файлов: {total_files:,}")

        # проверяем целостность
        with zipfile.ZipFile(output_zip_path, 'r') as check_zip:
            bad_file = check_zip.testzip()
            if bad_file:
                print(f"ошибка в файле: {bad_file}")
            else:
                print(f"zip корректен")

        # очищаем временную папку
        shutil.rmtree(temp_dir)

        return total_files

images_parts = [
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART1.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART2.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART3.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART4.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART5.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART6.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART7.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART8.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART9.zip",
    "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images_PART10.zip",
]

output_images = "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_images.zip"

confirmation = input("продолжить? (да/нет): ")
if confirmation.lower() in ['да']:
    total = merge_and_clean_images(images_parts, output_images)
    print(f"\n готово, обработано {total:,} файлов")
else:
    print("отменено пользователем")

In [ ]:
gt_zips = [
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART1.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART2.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART3.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART4.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART5.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART6.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART7.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART8.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART9.zip",
    "/content/drive/MyDrive/Kabard_NoOCR_balanced_sampled_generated_gt_PART10.zip",
]

output_zip = "/content/drive/MyDrive/Adyghe_NoOCR_balanced_sampled_generated_gt.zip"

total_files = 0
all_filenames = []

# создаем итоговый ZIP
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as out_zip:

    # обрабатываем каждый PART
    for i, zip_path in enumerate(gt_zips, 1):
        print(f"\nчасть {i}/{len(gt_zips)}: {os.path.basename(zip_path)}")

        if not os.path.exists(zip_path):
            print(f"файл не найден!")
            continue

        with zipfile.ZipFile(zip_path, 'r') as in_zip:
            files = in_zip.namelist()
            print(f"найдено файлов: {len(files)}")

            for filename in tqdm(files, desc="копирование", leave=False):
                content = in_zip.read(filename)
                out_zip.writestr(filename, content)
                all_filenames.append(filename)

            total_files = total_files + len(files)

print(f"\nобщее количество файлов: {total_files:,}")

# ожидаемое количество: 100,000 текстов х 12 вариаций = 1,200,000
expected = 100000 * 12  # 1,200,000

if total_files == expected:
    print(f"количество совпадает, все {expected:,} файлов на месте")
else:
    print(f"количество не совпадает! разница: {total_files - expected:,}")

# проверяем уникальность имен
unique_files = len(set(all_filenames))
print(f"\nуникальных имен: {unique_files:,}")
if unique_files == total_files:
    print("все имена уникальны")
else:
    print(f"есть дубликаты! потеряно {total_files - unique_files} файлов")

print(f"\nразмер итогового zip: {os.path.getsize(output_zip) / (1024*1024):.2f} MB")